# 02 — MCP Locally via stdio

**What you'll learn**
- The three MCP roles: **server**, **client**, **host**.
- What the `@mcp.tool()` decorator actually does.
- How tool discovery works, and why it matters for LLMs.
- What stdio transport means and when to use it.
- How to wire a real FastMCP server to Claude Desktop or Cursor.

## Architecture

```text
+-----------------+        list_tools / call_tool         +-----------------+
| Host (AI app)   |  <----------------------------------> | MCP server      |
|  - LLM          |    stdin / stdout (JSON-RPC frames)   |  - tools        |
|  - planning     |                                        |  - business logic|
+--------+--------+                                        +--------+--------+
         |                                                          |
         |                                                          v
         |                                                  +---------------+
         +------- MCP client (talks the protocol) --------> | Fake CRM      |
                                                            +---------------+
```

**Roles you must keep separate in your head:**

| Role | Job | Example |
|---|---|---|
| Host | The user-facing AI app. Owns the LLM and the conversation. | Claude Desktop, Cursor, your custom agent |
| Client | The bit of code inside the host that speaks MCP. One client per server. | `mcp.client.stdio` |
| Server | A process that exposes tools and data. | Your `sales_mcp_server.py` |

**stdio transport** means the host spawns the server as a subprocess and they exchange JSON-RPC frames over stdin/stdout. No network, no ports, no auth needed. Perfect for local desktop AI apps.

## A minimal MCP-style server

We use a small simulation class so every cell runs without installing `mcp`. The real FastMCP code follows at the bottom of the notebook — the shape is identical.

In [ ]:
from typing import Callable, Any

class MiniMCPServer:
    """Tiny MCP-style server used for teaching.

    Real MCP defines a JSON-RPC protocol on top of a transport (stdio or HTTP).
    This class keeps only the three ideas you actually need to internalize:
      1. tools are registered with a name + description + schema
      2. a client can list them
      3. a client can call one by name with arguments
    """

    def __init__(self, name: str) -> None:
        self.name = name
        self._tools: dict[str, Callable[..., Any]] = {}
        self._descriptions: dict[str, str] = {}

    def tool(self, description: str = ""):
        """Decorator that registers a function as an MCP tool."""
        def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
            self._tools[func.__name__] = func
            self._descriptions[func.__name__] = description or (func.__doc__ or "").strip()
            return func
        return decorator

    def list_tools(self) -> list[dict]:
        """Discovery: what can I do?"""
        return [{"name": n, "description": self._descriptions[n]} for n in self._tools]

    def call_tool(self, name: str, arguments: dict) -> Any:
        """Execution: do the thing."""
        if name not in self._tools:
            raise ValueError(f"Unknown tool: {name}")
        return self._tools[name](**arguments)

## Server: state + tool registration

The decorator-based registration is the **whole point** of FastMCP. You write a Python function, slap `@mcp.tool()` on it, and the framework:
- exposes it under the name `your_function`
- uses the docstring as the description the LLM sees
- (in real MCP) generates a JSON Schema from your type hints

Notice the host code below never imports `create_contact` directly.

In [ ]:
# Fake in-memory CRM. In production this would be HubSpot, Salesforce, etc.
CONTACTS: dict = {}
TASKS: list = []
OSC_TEAM = [
    {"id": "osc_101", "name": "Ava OSC",  "last_assigned": 0},
    {"id": "osc_102", "name": "Ben OSC",  "last_assigned": 0},
    {"id": "osc_103", "name": "Cara OSC", "last_assigned": 0},
]
_assignment_counter = 0
print("Fake CRM ready. Contacts:", len(CONTACTS), "OSCs:", len(OSC_TEAM))

In [ ]:
server = MiniMCPServer("sales-tools")

@server.tool(description="Create a new contact in the CRM")
def create_contact(name: str, email: str) -> dict:
    contact_id = f"contact_{len(CONTACTS) + 1}"
    contact = {"id": contact_id, "name": name, "email": email, "owner_id": None}
    CONTACTS[contact_id] = contact
    return contact

@server.tool(description="Assign an OSC to a contact using round-robin")
def assign_osc(contact_id: str) -> dict:
    global _assignment_counter
    if contact_id not in CONTACTS:
        raise ValueError(f"Contact not found: {contact_id}")
    chosen = min(OSC_TEAM, key=lambda osc: osc["last_assigned"])
    _assignment_counter += 1
    chosen["last_assigned"] = _assignment_counter
    CONTACTS[contact_id]["owner_id"] = chosen["id"]
    return chosen

@server.tool(description="Create a follow-up task for a contact")
def create_followup_task(contact_id: str, note: str) -> dict:
    if contact_id not in CONTACTS:
        raise ValueError(f"Contact not found: {contact_id}")
    task = {"id": f"task_{len(TASKS) + 1}", "contact_id": contact_id,
            "note": note, "status": "open"}
    TASKS.append(task)
    return task

server.list_tools()

## Client: speaks the protocol

The client knows nothing about the tools. It can list them and call them by name. That's the entire surface area, and it's exactly what an LLM needs to be useful.

In [ ]:
class MiniMCPClient:
    """Talks to a MiniMCPServer. In real MCP this would speak JSON-RPC over a transport."""

    def __init__(self, server: MiniMCPServer):
        self._server = server

    def list_tools(self) -> list[dict]:
        return self._server.list_tools()

    def call_tool(self, name: str, arguments: dict):
        print(f"[client] -> {name}({arguments})")
        return self._server.call_tool(name, arguments)

client = MiniMCPClient(server)
for t in client.list_tools():
    print(t)

## Host: discovers, plans, calls

The host is the only piece that knows about the user, the LLM, and the conversation. It treats tools as a black-box list it received from `list_tools`.

In [ ]:
import os

def llm_plan(user_message: str) -> list[dict]:
    """Return the agent's tool-call plan.

    If OPENAI_API_KEY or ANTHROPIC_API_KEY is set, you would normally call the LLM
    here with the available tool schemas and let it choose. To keep this notebook
    100% runnable offline, we always return a deterministic mock plan. The shape
    is what a real LLM tool-use response would give you after parsing.

    Each step optionally has a "bind" name. Later steps reference earlier
    results with "$<bind>.<field>". This mirrors how real LLM tool-calling
    chains outputs across calls.
    """
    if os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"):
        # Real implementation hook — left as a comment so the example still runs
        # without a key. Sketch:
        #   from openai import OpenAI
        #   client = OpenAI()
        #   resp = client.chat.completions.create(model="gpt-4o", messages=[...], tools=[...])
        #   return parse_tool_calls(resp)
        pass

    return [
        {"tool": "create_contact",
         "args": {"name": "John Doe", "email": "john@example.com"},
         "bind": "contact"},
        {"tool": "assign_osc",
         "args": {"contact_id": "$contact.id"}},
        {"tool": "create_followup_task",
         "args": {"contact_id": "$contact.id",
                  "note": "Follow up with John Doe within 24 hours"}},
    ]


def resolve_placeholders(args: dict, bindings: dict) -> dict:
    """Replace $<bind>.<field> tokens with values from earlier tool results."""
    resolved = {}
    for k, v in args.items():
        if isinstance(v, str) and v.startswith("$"):
            ref, field = v[1:].split(".", 1)
            if ref not in bindings:
                raise ValueError(f"unknown binding: {ref}")
            resolved[k] = bindings[ref][field]
        else:
            resolved[k] = v
    return resolved

In [ ]:
def host_agent(user_message: str) -> list[dict]:
    # 1. Discovery
    tools = client.list_tools()
    tool_names = [t["name"] for t in tools]
    print(f"[host] tools available: {tool_names}")

    # 2. Planning (real LLM would see `tools` and pick)
    plan = llm_plan(user_message)

    # 3. Execution
    results = []
    bindings: dict = {}
    for step in plan:
        if step["tool"] not in tool_names:
            raise ValueError(f"LLM hallucinated unknown tool: {step['tool']}")
        args = resolve_placeholders(step["args"], bindings)
        result = client.call_tool(step["tool"], args)
        results.append({"tool": step["tool"], "result": result})
        if "bind" in step:
            bindings[step["bind"]] = result
    return results

host_agent("Create a contact for John and assign an OSC")

## Mini test

In [ ]:
assert len(client.list_tools()) == 3
assert CONTACTS["contact_1"]["owner_id"].startswith("osc_")
assert TASKS[0]["status"] == "open"
print("ok")

## The real thing — a FastMCP server file

Everything above was simulated so the notebook would run anywhere. Here is the equivalent **real** server. Save it as `sales_mcp_server.py`, install the SDK with `pip install mcp`, and run `python sales_mcp_server.py`.

Compare it to the simulation — the decorator-based shape is identical, which is the whole reason FastMCP feels easy.

In [ ]:
REAL_MCP_SERVER = r'''# sales_mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("sales-tools")

CONTACTS, TASKS = {}, []
OSC_TEAM = [
    {"id": "osc_101", "name": "Ava OSC",  "last_assigned": 0},
    {"id": "osc_102", "name": "Ben OSC",  "last_assigned": 0},
    {"id": "osc_103", "name": "Cara OSC", "last_assigned": 0},
]
_counter = 0


@mcp.tool()
def create_contact(name: str, email: str) -> dict:
    """Create a new contact in the CRM."""
    cid = f"contact_{len(CONTACTS) + 1}"
    CONTACTS[cid] = {"id": cid, "name": name, "email": email, "owner_id": None}
    return CONTACTS[cid]


@mcp.tool()
def assign_osc(contact_id: str) -> dict:
    """Round-robin assign an OSC."""
    global _counter
    chosen = min(OSC_TEAM, key=lambda o: o["last_assigned"])
    _counter += 1
    chosen["last_assigned"] = _counter
    CONTACTS[contact_id]["owner_id"] = chosen["id"]
    return chosen


@mcp.tool()
def create_followup_task(contact_id: str, note: str) -> dict:
    """Create an open follow-up task."""
    task = {"id": f"task_{len(TASKS) + 1}", "contact_id": contact_id,
            "note": note, "status": "open"}
    TASKS.append(task)
    return task


if __name__ == "__main__":
    # Default transport is stdio — perfect for local desktop apps.
    mcp.run()
'''
print(REAL_MCP_SERVER)

## Connecting to it from Python

The host spawns the server process and talks to it via the Python SDK's stdio client.

In [ ]:
REAL_STDIO_CLIENT = r'''# host_client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command="python",
    args=["sales_mcp_server.py"],
)

async def main():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("tools:", [t.name for t in tools.tools])

            contact = await session.call_tool(
                "create_contact",
                {"name": "John Doe", "email": "john@example.com"},
            )
            print("contact:", contact)

asyncio.run(main())
'''
print(REAL_STDIO_CLIENT)

## Connecting to it from Claude Desktop

Add this to `~/Library/Application Support/Claude/claude_desktop_config.json` (macOS). After restarting Claude Desktop, the tools show up in the UI and the model can call them in any chat.

In [ ]:
CLAUDE_DESKTOP_CONFIG = r'''{
  "mcpServers": {
    "sales-tools": {
      "command": "python",
      "args": ["/absolute/path/to/sales_mcp_server.py"]
    }
  }
}
'''
print(CLAUDE_DESKTOP_CONFIG)

## Key takeaway

MCP buys you **three boundaries**: the host owns the LLM, the client owns the protocol, the server owns the tools. The stdio transport is the quickest way to feel that separation — no network, no auth, just a subprocess. Once you've held this shape in your head, HTTP (notebook 03) is a small step.